# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Frequency Domain Analysis and Digital Filtering
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/04_FrequencyDomainFiltering.ipynb)


**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
This interactive computational laboratory provides an end-to-end, rigorous treatment of **2D Fourier Transform Theory, Spectral Analysis, and Frequency-Domain Digital Filtering**.

By completing this notebook, you will master:
1. **1D & 2D Fourier Fundamentals**: Euler's formula, orthogonal projections, magnitude vs. phase spectra, and the effect of noise on spectral distribution.
2. **Elementary 2D Gratings & Geometric Dualities**: Analyzing how spatial orientations, rotations, scaling, affine shearing, and non-rigid elastic deformations manifest in 2D Fourier space.
3. **Spectrum Centering & Dynamic Range Compression**: Mathematical basis of quadrant swapping ($(-1)^{x+y}$) and logarithmic spectral compression.
4. **Phase Dominance & Cross-Reconstruction**: The profound structural role of Fourier phase over magnitude in visual perception.
5. **Canonical Frequency-Domain Filtering Workflow**: Zero-padding ($P \ge 2M-1$), spatial modulation, spectral multiplication, and unpadding.
6. **Transfer Function Engineering**: Ideal, Butterworth, Gaussian, Band-Pass, Band-Reject, and Symmetric Multi-Notch Reject Filters.
7. **Periodic Interference Suppression**: Surgical frequency-domain notch filtering of multi-frequency noise.
8. **Homomorphic Filtering**: Illumination-reflectance separation via non-linear logarithmic transformation.
9. **Advanced Modern Spectral Techniques (14.1–14.9)**:
   - Optimal Wiener Deconvolution & Constrained Least Squares (CLS)
   - Phase Correlation for Sub-Pixel Registration
   - 2D Gabor Filter Banks for Directional Texture Analysis
   - Steerable Pyramids & Multiscale Oriented Decompositions
   - 2D Cepstral Quefrency Analysis
   - Fractional Fourier Transform (FrFT)
   - Total Variation (TV) Compressed Sensing Inpainting
   - High-Frequency Emphasis (HFE) Filtering
   - Fourier Ring Correlation (FRC) Quantitative Resolution Metrology

---


---
## SECTION 0: Environment Setup & Asset Verification

### Theoretical & Computational Context
Frequency-domain image processing operates on complex-valued spectral representations ($F(u,v) = R(u,v) + j I(u,v)$). Numerical computations require standard 64-bit floating-point arrays (`np.float64`) normalized to $[0, 1]$ to prevent integer overflow and truncation errors during forward and inverse Fast Fourier Transforms (FFT).

In this section, we import all required computational packages (`numpy`, `scipy`, `matplotlib`, `skimage`, `cv2`) and verify the local availability of benchmark test images (`ct_thorax.png`, `mri_t1n_brain.png`, `cameraman.tif`). If any assets are missing, high-fidelity synthetic/radiological fallbacks are automatically generated.


In [ ]:
# Section 0: Environment Setup & Asset Verification
import os
import sys
import urllib.request
import numpy as np
import scipy.ndimage as ndimage
import scipy.signal as signal
from scipy import fftpack
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from skimage import data, color, transform, filters, exposure, util
from PIL import Image

# Configure Matplotlib styling for high-resolution publication-quality plots
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9.5
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10.5
plt.rcParams['xtick.labelsize'] = 8.5
plt.rcParams['ytick.labelsize'] = 8.5
plt.rcParams['image.cmap'] = 'gray'

# 1. Ensure target directory exists and download required image assets if missing
lab_dir = "./example_data"
os.makedirs(lab_dir, exist_ok=True)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Astarakee/isa-su/main/labs/example_data"
ASSETS = {
    "cameraman.tif": f"{GITHUB_RAW_BASE}/cameraman.tif",
    "ct_thorax.png": f"{GITHUB_RAW_BASE}/ct_thorax.png",
    "mri_t1n_brain.png": f"{GITHUB_RAW_BASE}/mri_t1n_brain.png"
}

for fname, url in ASSETS.items():
    fpath = os.path.join(lab_dir, fname)
    if not os.path.exists(fpath):
        print(f"Downloading missing asset '{fname}' into {lab_dir}...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as resp, open(fpath, 'wb') as f:
                f.write(resp.read())
            print(f"Successfully downloaded '{fname}'")
        except Exception as e:
            print(f"Warning: Failed to download '{fname}': {e}")

def resolve_or_create_asset(filename, fallback_func):
    # Uses the downloaded repository asset if available; otherwise synthesizes a benchmark image
    fpath = os.path.join(lab_dir, filename)
    if os.path.exists(fpath):
        return fpath
    print(f"Warning: '{filename}' unavailable. Generating synthetic fallback asset.")
    img_data = fallback_func()
    if img_data.dtype != np.uint8:
        img_data = (255 * (img_data - img_data.min()) / (img_data.max() - img_data.min() + 1e-8)).astype(np.uint8)
    Image.fromarray(img_data).save(fpath)
    return fpath

# Fallback asset generators (used only if the repository download fails)
def _get_cameraman():
    return data.camera()

def _get_ct_thorax():
    try:
        raw = data.human_mitosis()
    except Exception:
        raw = data.camera()
    return raw

def _get_mri_brain():
    try:
        raw = data.brain()[10] # slice 10 of brain volumetric MRI
    except Exception:
        raw = data.coins()
    return raw

path_cameraman = resolve_or_create_asset('cameraman.tif', _get_cameraman)
path_ct = resolve_or_create_asset('ct_thorax.png', _get_ct_thorax)
path_mri = resolve_or_create_asset('mri_t1n_brain.png', _get_mri_brain)

print("Environment setup verified. All numerical libraries and imaging assets are ready.")

---
## SECTION 1: 1D Fourier Transform Fundamentals (Sine vs. Cosine Analysis)

### 1.1 Mathematical Formulation
The Continuous Fourier Transform (CFT) maps a continuous time-domain signal $x(t)$ to its continuous frequency spectrum $X(f)$:
$$X(f) = \int_{-\infty}^{\infty} x(t) e^{-j 2\pi f t} \, dt$$

For discrete, finite-length signals $x[n]$ of length $N$ sampled at sampling frequency $F_s$, the **Discrete Fourier Transform (DFT)** is defined as:
$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j \frac{2\pi}{N} k n}, \quad k = 0, 1, \dots, N-1$$

Using **Euler's Formula**, $e^{-j \theta} = \cos(\theta) - j \sin(\theta)$, we expand the kernel:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cos\left(\frac{2\pi}{N} kn\right) - j \sum_{n=0}^{N-1} x[n] \sin\left(\frac{2\pi}{N} kn\right) = \text{Re}(X[k]) + j \text{Im}(X[k])$$

### 1.2 Symmetry Properties of Pure Sine and Cosine Waves
1. **Pure Cosine Signal ($x_c[n] = \cos(2\pi f_0 n / F_s)$)**:
   Since cosine is an **even symmetric function** ($x_c[n] = x_c[-n]$), the imaginary sine projection vanishes identically ($\text{Im}(X_c) = 0$). The spectrum consists of two purely real, symmetric delta impulses:
   $$X_c[k] = \frac{N}{2} \left[ \delta(k - k_0) + \delta(k - (N - k_0)) \right], \quad \text{Phase } \angle X_c[k_0] = 0$$

2. **Pure Sine Signal ($x_s[n] = \sin(2\pi f_0 n / F_s)$)**:
   Since sine is an **odd anti-symmetric function** ($x_s[n] = -x_s[-n]$), the real cosine projection vanishes ($\text{Re}(X_s) = 0$). The spectrum consists of purely imaginary, anti-symmetric delta impulses:
   $$X_s[k] = \frac{N}{2j} \left[ \delta(k - k_0) - \delta(k - (N - k_0)) \right] = -j \frac{N}{2} \delta(k - k_0) + j \frac{N}{2} \delta(k - (N - k_0))$$
   $$\text{Phase } \angle X_s[k_0] = -\frac{\pi}{2} \text{ radians } (-90^\circ)$$

### 1.3 Magnitude and Phase Spectra
- **Magnitude Spectrum**: $|X[k]| = \sqrt{\text{Re}(X[k])^2 + \text{Im}(X[k])^2}$ (measures energy/amplitude at frequency bin $k$).
- **Phase Spectrum**: $\angle X[k] = \text{atan2}\big(\text{Im}(X[k]), \text{Re}(X[k])\big)$ (measures the temporal alignment / time shift).


In [ ]:
# Cell 1.1: Time-Domain Signal Generation (Pure Sine vs. Cosine)
Fs = 500.0          # Sampling frequency (Hz)
T = 1.0 / Fs        # Sampling interval (seconds)
N = 500             # Total number of sample points
t = np.arange(N) * T # Time vector [0, 1) seconds
f0 = 10.0           # Fundamental frequency of tone (Hz)

# Generate pure sine and cosine signals
sig_cos = np.cos(2.0 * np.pi * f0 * t)
sig_sin = np.sin(2.0 * np.pi * f0 * t)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(t[:100], sig_cos[:100], 'b-', linewidth=1.8, label=r'$x_c(t) = \cos(2\pi \cdot 10 t)$')
axes[0].set_title("Time Domain: Pure Cosine Signal ($f_0 = 10$ Hz)", fontweight='bold')
axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='upper right')

axes[1].plot(t[:100], sig_sin[:100], 'r-', linewidth=1.8, label=r'$x_s(t) = \sin(2\pi \cdot 10 t)$')
axes[1].set_title("Time Domain: Pure Sine Signal ($f_0 = 10$ Hz)", fontweight='bold')
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Amplitude")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 1.2: 1D Fast Fourier Transform, Magnitude, and Phase Computation
# Compute DFT via FFT
X_cos = np.fft.fft(sig_cos)
X_sin = np.fft.fft(sig_sin)

# Two-sided centered frequency axis
freqs = np.fft.fftshift(np.fft.fftfreq(N, d=T))

# Centered complex spectra
X_cos_shifted = np.fft.fftshift(X_cos)
X_sin_shifted = np.fft.fftshift(X_sin)

# Magnitude spectra (normalized)
mag_cos = np.abs(X_cos_shifted) / N
mag_sin = np.abs(X_sin_shifted) / N

# Phase spectra (mask negligible numerical noise below threshold)
thresh = 1e-3
phase_cos = np.angle(X_cos_shifted)
phase_cos[mag_cos < thresh] = 0.0

phase_sin = np.angle(X_sin_shifted)
phase_sin[mag_sin < thresh] = 0.0

fig, axes = plt.subplots(2, 2, figsize=(14, 6))

# Row 1: Magnitude Spectra
axes[0, 0].stem(freqs, mag_cos, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[0, 0].set_title("Cosine Magnitude Spectrum $|X_c(f)|$", fontweight='bold')
axes[0, 0].set_xlim(-30, 30)
axes[0, 0].set_ylabel("Normalized Magnitude")
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

axes[0, 1].stem(freqs, mag_sin, linefmt='r-', markerfmt='ro', basefmt='k-')
axes[0, 1].set_title("Sine Magnitude Spectrum $|X_s(f)|$", fontweight='bold')
axes[0, 1].set_xlim(-30, 30)
axes[0, 1].set_ylabel("Normalized Magnitude")
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

# Row 2: Phase Spectra
axes[1, 0].stem(freqs, phase_cos, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[1, 0].set_title(r"Cosine Phase Spectrum $\angle X_c(f)$ [$\theta = 0$ rad]", fontweight='bold')
axes[1, 0].set_xlim(-30, 30)
axes[1, 0].set_ylim(-np.pi, np.pi)
axes[1, 0].set_xlabel("Frequency (Hz)")
axes[1, 0].set_ylabel("Phase (radians)")
axes[1, 0].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1, 0].set_yticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

axes[1, 1].stem(freqs, phase_sin, linefmt='r-', markerfmt='ro', basefmt='k-')
axes[1, 1].set_title(r"Sine Phase Spectrum $\angle X_s(f)$ [$\theta = \mp \pi/2$ rad]", fontweight='bold')
axes[1, 1].set_xlim(-30, 30)
axes[1, 1].set_ylim(-np.pi, np.pi)
axes[1, 1].set_xlabel("Frequency (Hz)")
axes[1, 1].set_ylabel("Phase (radians)")
axes[1, 1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1, 1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


### Cell 1.3: Pedagogical Discussion & Theoretical Insights
1. **Magnitude Equivalence**: Both signals possess **identical magnitude spectra** ($|X(f)| = 0.5$ at $f = \pm 10\text{ Hz}$), proving that magnitude alone only reveals **what frequencies are present**, but is completely blind to **temporal positioning**.
2. **Phase Orthogonality**: The cosine wave exhibits $0\text{ rad}$ phase at $f_0 = +10\text{ Hz}$, while the sine wave exhibits $-\frac{\pi}{2}\text{ rad}$ ($-90^\circ$) phase. This $-\pi/2$ phase shift corresponds to the exact time shift $\Delta t = \frac{\pi/2}{2\pi f_0} = \frac{1}{4f_0} = 25\text{ ms}$ between $\cos(2\pi f_0 t)$ and $\sin(2\pi f_0 t) = \cos(2\pi f_0 t - \pi/2)$.


---
## SECTION 2: Spectral Analysis of Noisy 1D Signals

### 2.1 Additive White Gaussian Noise (AWGN) in Frequency Domain
When a continuous signal $s(t)$ is corrupted by zero-mean Additive White Gaussian Noise $\eta(t) \sim \mathcal{N}(0, \sigma^2)$, the observed signal is:
$$x(t) = s(t) + \eta(t)$$

By the linearity of the Fourier Transform:
$$X(f) = S(f) + N(f)$$

- **Spectral Power Density of White Noise**: The theoretical Power Spectral Density (PSD) of white noise is flat across all frequency bins: $S_{\eta\eta}(f) = \sigma^2$.
- **Energy Preservation (Parseval's Theorem)**:
$$\sum_{n=0}^{N-1} |x[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |X[k]|^2$$
While the deterministic sinusoidal signal energy remains concentrated inside two narrow spectral bins ($k = \pm k_0$), the random noise energy $\sigma^2$ is uniformly dispersed across all $N$ frequency bins.


In [ ]:
# Cell 2.1: Noise Injection into 1D Signals
np.random.seed(42)
noise_sigma = 0.85
noise = np.random.normal(0, noise_sigma, N)
sig_noisy = sig_sin + noise

# Cell 2.2: Compute Spectrum of Noisy Signal
X_noisy = np.fft.fft(sig_noisy)
X_noisy_shifted = np.fft.fftshift(X_noisy)
mag_noisy = np.abs(X_noisy_shifted) / N
phase_noisy = np.angle(X_noisy_shifted)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (A) Noisy Time Waveform
axes[0].plot(t[:150], sig_noisy[:150], color='darkorange', linewidth=1.2, label='Noisy: $s(t) + \eta(t)$')
axes[0].plot(t[:150], sig_sin[:150], 'k--', linewidth=1.5, label='Clean Sine')
axes[0].set_title("(A) Time Domain: Signal with AWGN ($\sigma=0.85$)", fontweight='bold')
axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Amplitude")
axes[0].legend(loc='upper right', fontsize=8.5)
axes[0].grid(True, linestyle='--', alpha=0.6)

# (B) Noisy Magnitude Spectrum
axes[1].plot(freqs, mag_noisy, color='darkorange', linewidth=1.2)
axes[1].plot([-f0, f0], [0.5, 0.5], 'ro', markersize=6, label='Harmonic Peaks ($f_0 = \pm 10$ Hz)')
axes[1].axhline(y=noise_sigma / np.sqrt(N), color='k', linestyle=':', label=r'Expected Noise Floor $\sigma/\sqrt{N}$')
axes[1].set_title("(B) Shifted Magnitude Spectrum $|X_{\mathrm{noisy}}(f)|$", fontweight='bold')
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Normalized Magnitude")
axes[1].set_xlim(-100, 100)
axes[1].legend(loc='upper right', fontsize=8.5)
axes[1].grid(True, linestyle='--', alpha=0.6)

# (C) Noisy Phase Spectrum
axes[2].plot(freqs, phase_noisy, color='darkorange', linewidth=0.8)
axes[2].set_title(r"(C) Shifted Phase Spectrum $\angle X_{\mathrm{noisy}}(f)$", fontweight='bold')
axes[2].set_xlabel("Frequency (Hz)")
axes[2].set_ylabel("Phase (radians)")
axes[2].set_xlim(-100, 100)
axes[2].set_ylim(-np.pi, np.pi)
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


---
## SECTION 3: 2D DFT of Elementary 2D Gratings (Horizontal Patterns)

### 3.1 The 2D Discrete Fourier Transform
Let $f(x,y)$ be a 2D digital image of size $M \times N$, where $x \in \{0, \dots, M-1\}$ denotes row coordinates (vertical spatial axis) and $y \in \{0, \dots, N-1\}$ denotes column coordinates (horizontal spatial axis).

The **2D Forward Discrete Fourier Transform** is:
$$F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y) \exp\left[ -j 2\pi \left( \frac{ux}{M} + \frac{vy}{N} \right) \right]$$
where $u \in \{0, \dots, M-1\}$ is vertical spatial frequency, and $v \in \{0, \dots, N-1\}$ is horizontal spatial frequency.

### 3.2 Spectral Analysis of Horizontal Gratings
Consider a horizontal grating pattern $f_h(x,y) = \cos(2\pi f_0 x / M)$. 
- Along rows ($x$-direction), intensity oscillates at frequency $f_0$.
- Along columns ($y$-direction), intensity is strictly constant ($\frac{\partial f}{\partial y} = 0$).

Substituting $f_h(x,y)$ into the 2D DFT:
$$F_h(u,v) = \left[ \sum_{x=0}^{M-1} \cos\left(\frac{2\pi f_0 x}{M}\right) e^{-j \frac{2\pi ux}{M}} \right] \cdot \left[ \sum_{y=0}^{N-1} e^{-j \frac{2\pi vy}{N}} \right]$$
The second summation evaluates to the discrete Kronecker delta $N \cdot \delta(v)$. Therefore, **horizontal spatial stripes generate spectral impulse peaks strictly along the vertical frequency axis $u$ (at $v=0$)**.


In [ ]:
# Section 3: 2D DFT of Horizontal Grating Pattern
M_dim, N_dim = 256, 256
y_grid, x_grid = np.meshgrid(np.arange(N_dim), np.arange(M_dim))

# Spatial frequency f0 = 16 cycles per image dimension
f0_cycles = 16
grating_horiz = 0.5 + 0.5 * np.cos(2.0 * np.pi * f0_cycles * x_grid / M_dim)

# 2D DFT and Centering
F_horiz = np.fft.fft2(grating_horiz)
F_horiz_shifted = np.fft.fftshift(F_horiz)
mag_horiz_log = np.log1p(np.abs(F_horiz_shifted))
phase_horiz = np.angle(F_horiz_shifted)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

im0 = axes[0].imshow(grating_horiz, cmap='gray')
axes[0].set_title(r"(A) Spatial Domain: Horizontal Grating $f_h(x,y)$", fontweight='bold')
axes[0].set_xlabel("Column $y$ (horizontal)")
axes[0].set_ylabel("Row $x$ (vertical)")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(mag_horiz_log, cmap='magma', extent=[-N_dim//2, N_dim//2, -M_dim//2, M_dim//2])
axes[1].set_title(r"(B) Centered Log-Magnitude $\log(1 + |F_h(u,v)|)$", fontweight='bold')
axes[1].set_xlabel("Horizontal Frequency $v$ (cycles/image)")
axes[1].set_ylabel("Vertical Frequency $u$ (cycles/image)")
axes[1].axhline(0, color='cyan', linestyle=':', alpha=0.5)
axes[1].axvline(0, color='cyan', linestyle=':', alpha=0.5)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(phase_horiz, cmap='twilight', extent=[-N_dim//2, N_dim//2, -M_dim//2, M_dim//2])
axes[2].set_title(r"(C) Phase Spectrum $\angle F_h(u,v)$ (radians)", fontweight='bold')
axes[2].set_xlabel("Horizontal Frequency $v$")
axes[2].set_ylabel("Vertical Frequency $u$")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


---
## SECTION 4: 2D DFT of Vertical Grating Patterns

### 4.1 Spatial-Frequency Orthogonal Duality
Conversely, consider a vertical grating pattern $f_v(x,y) = \cos(2\pi f_0 y / N)$.
- Along rows ($x$-direction), intensity is constant ($\frac{\partial f}{\partial x} = 0$).
- Along columns ($y$-direction), intensity oscillates at frequency $f_0$.

The 2D DFT decomposes as:
$$F_v(u,v) = \left[ \sum_{x=0}^{M-1} e^{-j \frac{2\pi ux}{M}} \right] \cdot \left[ \sum_{y=0}^{N-1} \cos\left(\frac{2\pi f_0 y}{N}\right) e^{-j \frac{2\pi vy}{N}} \right] = M \delta(u) \cdot \frac{N}{2}\big[\delta(v - f_0) + \delta(v + f_0)\big]$$
Thus, **vertical spatial stripes generate spectral impulse peaks strictly along the horizontal frequency axis $v$ (at $u=0$)**.


In [ ]:
# Section 4: 2D DFT of Vertical Grating Pattern
grating_vert = 0.5 + 0.5 * np.cos(2.0 * np.pi * f0_cycles * y_grid / N_dim)

# 2D DFT and Centering
F_vert = np.fft.fft2(grating_vert)
F_vert_shifted = np.fft.fftshift(F_vert)
mag_vert_log = np.log1p(np.abs(F_vert_shifted))
phase_vert = np.angle(F_vert_shifted)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

im0 = axes[0].imshow(grating_vert, cmap='gray')
axes[0].set_title(r"(A) Spatial Domain: Vertical Grating $f_v(x,y)$", fontweight='bold')
axes[0].set_xlabel("Column $y$ (horizontal)")
axes[0].set_ylabel("Row $x$ (vertical)")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(mag_vert_log, cmap='magma', extent=[-N_dim//2, N_dim//2, -M_dim//2, M_dim//2])
axes[1].set_title(r"(B) Centered Log-Magnitude $\log(1 + |F_v(u,v)|)$", fontweight='bold')
axes[1].set_xlabel("Horizontal Frequency $v$ (cycles/image)")
axes[1].set_ylabel("Vertical Frequency $u$ (cycles/image)")
axes[1].axhline(0, color='cyan', linestyle=':', alpha=0.5)
axes[1].axvline(0, color='cyan', linestyle=':', alpha=0.5)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(phase_vert, cmap='twilight', extent=[-N_dim//2, N_dim//2, -M_dim//2, M_dim//2])
axes[2].set_title(r"(C) Phase Spectrum $\angle F_v(u,v)$ (radians)", fontweight='bold')
axes[2].set_xlabel("Horizontal Frequency $v$")
axes[2].set_ylabel("Vertical Frequency $u$")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


---
## SECTION 5: Rotation Property of the 2D Fourier Transform

### 5.1 Mathematical Theorem
Represent spatial coordinates in polar form: $x = r\cos\theta, y = r\sin\theta$, and frequency coordinates as $u = \rho\cos\phi, v = \rho\sin\phi$.

If a spatial image $f(r, \theta)$ is rotated by angle $\theta_0$ ($f(r, \theta + \theta_0)$), its 2D Fourier transform rotates by the **exact same angle $\theta_0$**:
$$\mathcal{F}\big\{ f(r, \theta + \theta_0) \big\} = F(\rho, \phi + \theta_0)$$

This property proves that Fourier spectrum orientation is covariant with spatial structure orientation.


In [ ]:
# Section 5: Rotation Property Demonstration
rot_angle_deg = 15.0 # 15 degrees rotation

# Rotate spatial gratings
grating_horiz_rot = transform.rotate(grating_horiz, angle=rot_angle_deg, mode='wrap')
grating_vert_rot = transform.rotate(grating_vert, angle=rot_angle_deg, mode='wrap')

# Compute rotated 2D DFTs
F_h_rot = np.fft.fftshift(np.fft.fft2(grating_horiz_rot))
F_v_rot = np.fft.fftshift(np.fft.fft2(grating_vert_rot))

mag_h_rot = np.log1p(np.abs(F_h_rot))
mag_v_rot = np.log1p(np.abs(F_v_rot))

fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))

# Row 1: Horizontal Grating Rotation
axes[0, 0].imshow(grating_horiz, cmap='gray')
axes[0, 0].set_title(r"Original Horizontal ($0^\circ$)", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(mag_horiz_log, cmap='magma')
axes[0, 1].set_title(r"Original Spectrum ($0^\circ$)", fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(grating_horiz_rot, cmap='gray')
axes[0, 2].set_title(f"Rotated Grating (+{rot_angle_deg}$^\\circ$)", fontweight='bold', color='navy')
axes[0, 2].axis('off')

axes[0, 3].imshow(mag_h_rot, cmap='magma')
axes[0, 3].set_title(f"Rotated Spectrum (+{rot_angle_deg}$^\\circ$)", fontweight='bold', color='navy')
axes[0, 3].axis('off')

# Row 2: Vertical Grating Rotation
axes[1, 0].imshow(grating_vert, cmap='gray')
axes[1, 0].set_title(r"Original Vertical ($90^\circ$)", fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(mag_vert_log, cmap='magma')
axes[1, 1].set_title(r"Original Spectrum ($90^\circ$)", fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(grating_vert_rot, cmap='gray')
axes[1, 2].set_title(f"Rotated Grating (+{rot_angle_deg}$^\\circ$)", fontweight='bold', color='darkgreen')
axes[1, 2].axis('off')

axes[1, 3].imshow(mag_v_rot, cmap='magma')
axes[1, 3].set_title(f"Rotated Spectrum (+{rot_angle_deg}$^\\circ$)", fontweight='bold', color='darkgreen')
axes[1, 3].axis('off')

plt.suptitle("Figure 5.1: Proof of Fourier Rotation Covariance: Spatial Rotation Rotates the 2D Spectrum by the Identical Angle",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 6: Affine & Non-Rigid Geometric Transformations in Frequency Space

### 6.1 Mathematical Principles of Spatial Mappings
1. **Reciprocal Scaling Property**:
   $$f(ax, by) \Longleftrightarrow \frac{1}{|ab|} F\left(\frac{u}{a}, \frac{v}{b}\right)$$
   Spatial expansion ($a > 1$) causes spectral compression ($u/a$), while spatial compression ($a < 1$) causes spectral broadening.
2. **Spatial Reflection**:
   $$f(-x, y) \Longleftrightarrow F(-u, v), \qquad f(x, -y) \Longleftrightarrow F(u, -v)$$
3. **Affine Shear Transformation**: $\mathbf{x}' = \mathbf{A} \mathbf{x} \implies F'(\mathbf{u}) = \frac{1}{|\det \mathbf{A}|} F(\mathbf{A}^{-T} \mathbf{u})$.
4. **Non-Rigid Elastic Deformations**: Induce local harmonic phase modulations and spectral dispersion.


In [ ]:
# Cell 6.1: Chessboard Synthesis
def create_chessboard(dim=256, num_blocks=8):
    block_size = dim // num_blocks
    board = np.zeros((dim, dim), dtype=np.float64)
    for i in range(num_blocks):
        for j in range(num_blocks):
            if (i + j) % 2 == 0:
                board[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = 1.0
    return board

base_board = create_chessboard(256, 8)

# Cell 6.2: Apply Geometric Transformations
# 1. Scale
t_scale = transform.rescale(base_board, 0.6, mode='constant', anti_aliasing=True)
pad_h = (256 - t_scale.shape[0]) // 2
pad_w = (256 - t_scale.shape[1]) // 2
t_scale_padded = np.pad(t_scale, ((pad_h, 256 - t_scale.shape[0] - pad_h), (pad_w, 256 - t_scale.shape[1] - pad_w)), mode='constant')

# 2. Horizontal Flip
t_hflip = np.fliplr(base_board)

# 3. Vertical Flip
t_vflip = np.flipud(base_board)

# 4. Rotation 45 deg
t_rot45 = transform.rotate(base_board, 45, mode='constant')

# 5. Affine Shear
af_trans = transform.AffineTransform(shear=0.35)
t_shear = transform.warp(base_board, inverse_map=af_trans.inverse, mode='constant')

# 6. Non-Rigid Sinusoidal Elastic Warp
nr_y, nr_x = np.meshgrid(np.linspace(0, 1, 256), np.linspace(0, 1, 256))
warp_x = nr_x + 0.04 * np.sin(2 * np.pi * 3 * nr_y)
warp_y = nr_y + 0.04 * np.cos(2 * np.pi * 3 * nr_x)
coords = np.array([warp_x * 255, warp_y * 255])
t_nonrigid = ndimage.map_coordinates(base_board, coords, order=1, mode='reflect')

transforms_list = [
    ("Original", base_board),
    ("Scaled (0.6x)", t_scale_padded),
    ("H-Flip", t_hflip),
    ("V-Flip", t_vflip),
    ("Rotated 45°", t_rot45),
    ("Affine Shear", t_shear),
    ("Non-Rigid Warp", t_nonrigid)
]

# Cell 6.3: 2 x 7 Multi-Panel Visualization
fig, axes = plt.subplots(2, 7, figsize=(18, 5.5))

for idx, (title, img) in enumerate(transforms_list):
    # Spatial Plot
    axes[0, idx].imshow(img, cmap='gray')
    axes[0, idx].set_title(title, fontweight='bold', fontsize=9.5)
    axes[0, idx].axis('off')
    
    # 2D Spectrum
    F_spec = np.fft.fftshift(np.fft.fft2(img))
    mag_spec = np.log1p(np.abs(F_spec))
    axes[1, idx].imshow(mag_spec, cmap='magma')
    axes[1, idx].set_title(f"Spectrum\n[{title}]", fontsize=8.5)
    axes[1, idx].axis('off')

plt.suptitle("Figure 6.1: Comprehensive Geometric Transformation Suite in Spatial (Row 1) and Frequency (Row 2) Domains",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 7: 2D Spectrum Centering and Dynamic Range Compression

### 7.1 Mathematical Derivation of Spectrum Centering
By default, the Discrete Fourier Transform algorithm places the zero-frequency DC component $F(0,0)$ at the origin $(0,0)$ (top-left corner of the discrete array), with high frequencies in the center.

To center the spectrum so that DC resides at $(M/2, N/2)$, we use the **Frequency Shift Theorem**:
$$f(x,y) e^{j 2\pi (u_0 x / M + v_0 y / N)} \Longleftrightarrow F(u - u_0, v - v_0)$$
Setting $u_0 = M/2$ and $v_0 = N/2$:
$$e^{j 2\pi (x/2 + y/2)} = e^{j \pi (x + y)} = (-1)^{x+y}$$
$$\mathcal{F}\big\{ f(x,y) (-1)^{x+y} \big\} = F(u - M/2, v - N/2)$$

### 7.2 Logarithmic Dynamic Range Compression
Because natural scenes exhibit a high DC component that can be $10^4$ to $10^6$ times larger than high-frequency edge components, direct linear display renders all high-frequency diffraction lobes completely black. We apply **logarithmic compression**:
$$D(u,v) = c \cdot \log\big(1 + |F(u,v)|\big)$$


In [ ]:
# Section 7: Spectrum Centering and Dynamic Range Compression
square_img = np.zeros((256, 256), dtype=np.float64)
square_img[112:144, 112:144] = 1.0 # Centered 32x32 bright square

# Raw 2D FFT
F_raw = np.fft.fft2(square_img)
mag_uncentered_linear = np.abs(F_raw)
mag_centered_linear = np.abs(np.fft.fftshift(F_raw))
mag_centered_log = np.log1p(mag_centered_linear)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

axes[0].imshow(square_img, cmap='gray')
axes[0].set_title("(A) Input Spatial Object\n[32x32 Centered Square]", fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(mag_uncentered_linear, cmap='viridis')
axes[1].set_title("(B) Raw Uncentered Linear $|F|$\n[DC at Corner (0,0)]", fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(mag_centered_linear, cmap='viridis')
axes[2].set_title("(C) Centered Linear $|F|$\n[Lobes Obscured by High DC]", fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(mag_centered_log, cmap='magma')
axes[3].set_title("(D) Centered Log-Compressed $\log(1+|F|)$\n[2D Sinc Diffraction Pattern Revealed]", fontweight='bold', color='darkgreen')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


---
## SECTION 8: Invariance & Transformation Analysis on Centered Geometries

### 8.1 Shift Invariance of Magnitude Spectrum
When a spatial pattern is shifted by $(\Delta x, \Delta y)$, its Fourier transform acquires a linear phase shift while its **magnitude spectrum remains perfectly invariant**:
$$f(x - x_0, y - y_0) \Longleftrightarrow F(u,v) \exp\left[ -j 2\pi \left( \frac{ux_0}{M} + \frac{vy_0}{N} \right) \right]$$
$$\big| \mathcal{F}\{f(x - x_0, y - y_0)\} \big| = |F(u,v)|$$


In [ ]:
# Section 8: Invariance Analysis
sq_shifted = np.roll(np.roll(square_img, 45, axis=0), -50, axis=1) # Translation
sq_rot = transform.rotate(square_img, 30.0, mode='constant')       # Rotation 30 deg
sq_scaled = transform.rescale(square_img, 0.5, mode='constant')    # Scale 0.5x
sq_scaled_padded = np.pad(sq_scaled, ((64, 64), (64, 64)), mode='constant')

test_suite = [
    ("Centered Square", square_img),
    ("Translated (+45, -50)", sq_shifted),
    ("Rotated (30°)", sq_rot),
    ("Scaled (0.5x)", sq_scaled_padded)
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for idx, (label, img) in enumerate(test_suite):
    F_c = np.fft.fftshift(np.fft.fft2(img))
    mag_c = np.log1p(np.abs(F_c))
    phase_c = np.angle(F_c)
    
    axes[0, idx].imshow(img, cmap='gray')
    axes[0, idx].set_title(f"Spatial: {label}", fontweight='bold')
    axes[0, idx].axis('off')
    
    axes[1, idx].imshow(mag_c, cmap='magma')
    axes[1, idx].set_title(f"Log-Magnitude $|F|$\n[{label}]", fontsize=9)
    axes[1, idx].axis('off')
    
    axes[2, idx].imshow(phase_c, cmap='twilight')
    axes[2, idx].set_title(f"Phase $\\angle F$\n[{label}]", fontsize=9)
    axes[2, idx].axis('off')

plt.suptitle("Figure 8.1: Proof of Translation Invariance in Magnitude (Columns 1 vs. 2) and Reciprocal Scaling (Column 4)",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 9: Phase vs. Magnitude Dominance (Cross-Reconstruction Experiment)

### 9.1 Oppenheim's Phase Dominance Theorem
The polar representation of the 2D Fourier Transform separates amplitude and structure:
$$F(u,v) = |F(u,v)| \exp\big[ j \phi(u,v) \big]$$

- **Magnitude $|F(u,v)|$**: Encodes global spectral power, overall contrast, and dominant frequency orientations.
- **Phase $\phi(u,v)$**: Encodes the **precise spatial phase coherence** that constructs edges, corners, lines, and recognizable anatomical boundaries.

In this experiment, we combine the magnitude spectrum of Image A ($\text{CT Thorax}$) with the phase spectrum of Image B ($\text{MRI Brain}$), and vice versa:
$$\hat{F}_1(u,v) = |F_{\text{CT}}(u,v)| \cdot e^{j \phi_{\text{MRI}}(u,v)} \quad \Longrightarrow \quad \hat{f}_1(x,y) = \text{Re}\big\{\mathcal{F}^{-1}\{\hat{F}_1\}\big\}$$
$$\hat{F}_2(u,v) = |F_{\text{MRI}}(u,v)| \cdot e^{j \phi_{\text{CT}}(u,v)} \quad \Longrightarrow \quad \hat{f}_2(x,y) = \text{Re}\big\{\mathcal{F}^{-1}\{\hat{F}_2\}\big\}$$


In [ ]:
# Section 9: Phase vs. Magnitude Dominance Experiment
# Load and resize benchmark images to 256x256
img_ct_raw = Image.open(path_ct).convert('L').resize((256, 256))
img_mri_raw = Image.open(path_mri).convert('L').resize((256, 256))

img_ct = np.array(img_ct_raw, dtype=np.float64) / 255.0
img_mri = np.array(img_mri_raw, dtype=np.float64) / 255.0

# Forward 2D FFT
F_ct = np.fft.fft2(img_ct)
F_mri = np.fft.fft2(img_mri)

# Extract Magnitudes and Phases
mag_ct, phase_ct = np.abs(F_ct), np.angle(F_ct)
mag_mri, phase_mri = np.abs(F_mri), np.angle(F_mri)

# Synthesize Mixed Cross-Reconstructions
F_mix1 = mag_ct * np.exp(1j * phase_mri) # CT Magnitude + MRI Phase
F_mix2 = mag_mri * np.exp(1j * phase_ct) # MRI Magnitude + CT Phase

# Inverse 2D FFT
recon_mix1 = np.real(np.fft.ifft2(F_mix1))
recon_mix2 = np.real(np.fft.ifft2(F_mix2))

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(img_ct, cmap='gray')
axes[0, 0].set_title("(A) Original Image 1: CT Thorax", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_mri, cmap='gray')
axes[0, 1].set_title("(B) Original Image 2: MRI Brain", fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(recon_mix1, cmap='gray')
axes[1, 0].set_title("(C) Recon 1: $|F_{\mathrm{CT}}| \cdot e^{j \phi_{\mathrm{MRI}}}$\n[RESULT: MRI Anatomy is Clearly Visible!]",
                     fontweight='bold', color='darkred')
axes[1, 0].axis('off')

axes[1, 1].imshow(recon_mix2, cmap='gray')
axes[1, 1].set_title("(D) Recon 2: $|F_{\mathrm{MRI}}| \cdot e^{j \phi_{\mathrm{CT}}}$\n[RESULT: CT Anatomy is Clearly Visible!]",
                     fontweight='bold', color='darkblue')
axes[1, 1].axis('off')

plt.suptitle("Figure 9.1: Proof of Phase Dominance: Reconstructed Image Geometry is Governed Entirely by the Phase Spectrum",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 10: Canonical Frequency Domain Filtering Workflow

### 10.1 The 5-Step Filtering Pipeline
To perform linear spatial convolution via the Frequency Domain Convolution Theorem ($f * h \Longleftrightarrow F \cdot H$), we must prevent **circular convolution wraparound artifacts** caused by discrete periodicity.

The mathematically complete pipeline is:
1. **Zero-Padding**: Pad input image $f(x,y)$ of size $M \times N$ to size $P \times Q$, where:
   $$P \ge 2M - 1, \qquad Q \ge 2N - 1 \quad (\text{Standard Choice: } P = 2M, \, Q = 2N)$$
2. **Spatial Modulation & Forward DFT**: Compute centered 2D DFT $F_p(u,v) = \text{fftshift}(\text{fft2}(f_p(x,y)))$.
3. **Filter Transfer Function Multiplication**:
   $$G_p(u,v) = H(u,v) \cdot F_p(u,v)$$
4. **Inverse DFT**: Compute inverse transformed image $g_p(x,y) = \text{Re}\big\{\text{ifft2}(\text{ifftshift}(G_p(u,v)))\big\}$.
5. **Cropping**: Crop the top-left $M \times N$ region of $g_p(x,y)$ to extract final filtered result $g(x,y)$.


In [ ]:
# Section 10: Canonical Frequency Domain Filtering Engine
def frequency_domain_filter(img, H_uv):
    # Executes the canonical padded 2D frequency-domain filtering pipeline
    M, N = img.shape
    P, Q = 2 * M, 2 * N
    
    # 1. Zero-pad image to P x Q
    padded = np.zeros((P, Q), dtype=np.float64)
    padded[:M, :N] = img
    
    # 2. Forward 2D FFT and centering
    F = np.fft.fft2(padded)
    F_shift = np.fft.fftshift(F)
    
    # 3. Spectral multiplication
    G_shift = F_shift * H_uv
    
    # 4. Inverse centering and Inverse 2D FFT
    G = np.fft.ifftshift(G_shift)
    filtered_padded = np.real(np.fft.ifft2(G))
    
    # 5. Crop to original dimensions M x N
    filtered_img = filtered_padded[:M, :N]
    return filtered_img, F_shift, G_shift

# Demonstration on Cameraman with Gaussian LPF
img_cam_raw = Image.open(path_cameraman).convert('L').resize((256, 256))
img_cam = np.array(img_cam_raw, dtype=np.float64) / 255.0

# Add subtle Gaussian noise
np.random.seed(42)
img_cam_noisy = np.clip(img_cam + np.random.normal(0, 0.08, img_cam.shape), 0.0, 1.0)

# Construct Gaussian Low-Pass Filter on 2M x 2N grid
P_dim, Q_dim = 512, 512
u_coords = np.arange(P_dim) - P_dim / 2
v_coords = np.arange(Q_dim) - Q_dim / 2
U_mesh, V_mesh = np.meshgrid(v_coords, u_coords)
D_uv = np.sqrt(U_mesh**2 + V_mesh**2)

D0_cutoff = 40.0 # Cutoff frequency in pixels
H_glpf = np.exp(-(D_uv**2) / (2.0 * (D0_cutoff**2)))

# Execute Filter Engine
restored_img, F_inp_spec, G_out_spec = frequency_domain_filter(img_cam_noisy, H_glpf)

fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))

axes[0, 0].imshow(img_cam_noisy, cmap='gray')
axes[0, 0].set_title("(A) Input Noisy Image $f(x,y)$", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(np.log1p(np.abs(F_inp_spec)), cmap='magma')
axes[0, 1].set_title("(B) Input Spectrum $\log(1+|F(u,v)|)$", fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(H_glpf, cmap='viridis')
axes[0, 2].set_title(f"(C) Gaussian LPF $H(u,v)$ ($D_0 = {D0_cutoff}$)", fontweight='bold')
axes[0, 2].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(G_out_spec)), cmap='magma')
axes[1, 0].set_title("(D) Filtered Spectrum $\log(1+|G(u,v)|)$", fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(restored_img, cmap='gray')
axes[1, 1].set_title("(E) Restored Spatial Image $g(x,y)$", fontweight='bold', color='darkgreen')
axes[1, 1].axis('off')

# Difference residual
diff_residual = np.abs(img_cam_noisy - restored_img)
axes[1, 2].imshow(diff_residual, cmap='hot')
axes[1, 2].set_title("(F) Noise Removed Residual $|f - g|$", fontweight='bold')
axes[1, 2].axis('off')

plt.suptitle("Figure 10.1: Complete Canonical Frequency Domain Filtering Workflow (Padded Pipeline)",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 11: Transfer Function Topography & Radial Profiles

### 11.1 Mathematical Filter Formulations
Let Euclidean frequency distance from DC be:
$$D(u,v) = \sqrt{(u - P/2)^2 + (v - Q/2)^2}$$

1. **Ideal Low-Pass Filter (ILPF)**:
   $$H_{\text{ILPF}}(u,v) = \begin{cases} 1 & \text{if } D(u,v) \le D_0 \\ 0 & \text{if } D(u,v) > D_0 \end{cases}$$
   *(Note: Sharp step discontinuity causes severe spatial ringing sinc ripples - Gibbs Phenomenon).*
2. **Butterworth Low-Pass Filter (BLPF)** of order $n$:
   $$H_{\text{BLPF}}(u,v) = \frac{1}{1 + \left( \frac{D(u,v)}{D_0} \right)^{2n}}$$
3. **Gaussian Low-Pass Filter (GLPF)**:
   $$H_{\text{GLPF}}(u,v) = \exp\left( -\frac{D^2(u,v)}{2D_0^2} \right)$$
   *(Note: Fourier transform of a Gaussian is another Gaussian; completely zero ringing artifacts).*
4. **Gaussian Band-Reject Filter (GBRF)** with bandwidth $W$:
   $$H_{\text{GBRF}}(u,v) = 1 - \exp\left[ -\left( \frac{D^2(u,v) - D_0^2}{D(u,v) \cdot W} \right)^2 \right]$$


In [ ]:
# Section 11: Radial Profiles and 3D Filter Topography
r_dist = np.linspace(0, 100, 400)
D0_val = 35.0
W_val = 20.0
n_order = 2

# 1D Radial Profiles
h_ilpf = (r_dist <= D0_val).astype(float)
h_blpf = 1.0 / (1.0 + (r_dist / D0_val)**(2 * n_order))
h_glpf = np.exp(-(r_dist**2) / (2.0 * D0_val**2))

h_ihpf = 1.0 - h_ilpf
h_bhpf = 1.0 - h_blpf
h_ghpf = 1.0 - h_glpf

h_gbrf = 1.0 - np.exp(-((r_dist**2 - D0_val**2) / (r_dist * W_val + 1e-6))**2)
h_gbpf = 1.0 - h_gbrf

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(r_dist, h_ilpf, 'k--', label='Ideal LPF')
axes[0].plot(r_dist, h_blpf, 'b-', label='Butterworth LPF ($n=2$)')
axes[0].plot(r_dist, h_glpf, 'r-', label='Gaussian LPF')
axes[0].set_title("(A) Low-Pass Filter (LPF) Radial Profiles", fontweight='bold')
axes[0].set_xlabel("Radial Frequency Distance $D(u,v)$ (px)")
axes[0].set_ylabel("Gain $H(D)$")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

axes[1].plot(r_dist, h_ihpf, 'k--', label='Ideal HPF')
axes[1].plot(r_dist, h_bhpf, 'b-', label='Butterworth HPF ($n=2$)')
axes[1].plot(r_dist, h_ghpf, 'r-', label='Gaussian HPF')
axes[1].set_title("(B) High-Pass Filter (HPF) Radial Profiles", fontweight='bold')
axes[1].set_xlabel("Radial Frequency Distance $D(u,v)$ (px)")
axes[1].set_ylabel("Gain $H(D)$")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

axes[2].plot(r_dist, h_gbrf, 'm-', label='Gaussian Band-Reject')
axes[2].plot(r_dist, h_gbpf, 'g-', label='Gaussian Band-Pass')
axes[2].set_title(f"(C) Band Filter Profiles ($D_0={D0_val}, W={W_val}$)", fontweight='bold')
axes[2].set_xlabel("Radial Frequency Distance $D(u,v)$ (px)")
axes[2].set_ylabel("Gain $H(D)$")
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Cell 11.2: 3D Surface Topography Plots of Transfer Functions
fig = plt.figure(figsize=(16, 4.5))

grid_sub = 128
u_sub = np.linspace(-64, 64, grid_sub)
v_sub = np.linspace(-64, 64, grid_sub)
U_s, V_s = np.meshgrid(u_sub, v_sub)
D_s = np.sqrt(U_s**2 + V_s**2)

# Compute 3D filter surfaces
surf_glpf = np.exp(-(D_s**2) / (2.0 * 25.0**2))
surf_bhpf = 1.0 / (1.0 + (25.0 / (D_s + 1e-6))**(2 * 2))
surf_gbpf = np.exp(-((D_s**2 - 35.0**2) / (D_s * 15.0 + 1e-6))**2)

surfs = [
    ("Gaussian Low-Pass ($D_0=25$)", surf_glpf),
    ("Butterworth High-Pass ($D_0=25, n=2$)", surf_bhpf),
    ("Gaussian Band-Pass ($D_0=35, W=15$)", surf_gbpf)
]

for idx, (title, surf_data) in enumerate(surfs):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    surf_plot = ax.plot_surface(U_s, V_s, surf_data, cmap='viridis', edgecolor='none', alpha=0.9)
    ax.set_title(title, fontweight='bold', fontsize=9.5)
    ax.set_xlabel('$u$', fontsize=8)
    ax.set_ylabel('$v$', fontsize=8)
    ax.set_zlabel('$H(u,v)$', fontsize=8)
    ax.view_init(elev=35, azim=-55)

plt.suptitle("Figure 11.1: 3D Surface Topography of Canonical 2D Frequency-Domain Transfer Functions",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 12: Periodic Noise Suppression via Frequency-Domain Notch Filtering

### 12.1 Mathematical Mechanics of Multi-Notch Reject Filters
When a sensor acquisition is corrupted by electrical line interference or mechanical vibrations, the noise manifests as 2D sinusoidal interference:
$$\eta_{\text{periodic}}(x,y) = \sum_{k=1}^K A_k \sin\left( 2\pi \left( \frac{u_k x}{M} + \frac{v_k y}{N} \right) \right)$$
In Fourier space, this generates $K$ pairs of discrete symmetric conjugate impulse spikes at coordinates $(\pm u_k, \pm v_k)$.

Spatial linear smoothing filters (Gaussian, Box, Median) cannot isolate these discrete frequencies and instead severely blur legitimate diagnostic structures. 

A **Butterworth Notch Reject Filter** centered at conjugate notch coordinates $\{(\pm u_k, \pm v_k)\}_{k=1}^K$ with radius $D_0$ is defined as:
$$H_{\text{NR}}(u,v) = \prod_{k=1}^K \frac{1}{1 + \left( \frac{D_0^2}{D_k(u,v) \cdot D_{-k}(u,v)} \right)^n}$$
where:
$$D_k(u,v) = \sqrt{(u - P/2 - u_k)^2 + (v - Q/2 - v_k)^2}$$
$$D_{-k}(u,v) = \sqrt{(u - P/2 + u_k)^2 + (v - Q/2 + v_k)^2}$$


In [ ]:
# Section 12: Periodic Noise Suppression via Notch Filtering
# Load CT Thorax slice
img_ct_base = np.array(Image.open(path_ct).convert('L').resize((256, 256)), dtype=np.float64) / 255.0
M_ct, N_ct = img_ct_base.shape

# Inject Multi-Frequency Sinusoidal Interference
y_c, x_c = np.meshgrid(np.arange(N_ct), np.arange(M_ct))
noise_periodic = (0.28 * np.sin(2.0 * np.pi * 32 * x_c / M_ct + 2.0 * np.pi * 32 * y_c / N_ct) +
                  0.22 * np.cos(2.0 * np.pi * 48 * x_c / M_ct - 2.0 * np.pi * 24 * y_c / N_ct))
img_ct_corrupt = np.clip(img_ct_base + noise_periodic, 0.0, 1.0)

# Construct Butterworth Notch Filter on Padded Grid
P_c, Q_c = 2 * M_ct, 2 * N_ct
u_grid_c = np.arange(P_c) - P_c / 2
v_grid_c = np.arange(Q_c) - Q_c / 2
U_c, V_c = np.meshgrid(v_grid_c, u_grid_c)

# Notch Coordinates (scaled to padded grid: 2 * spatial frequency)
notch_centers = [
    (64, 64),   # Pair 1 (+u0, +v0)
    (96, -48)   # Pair 2 (+u1, -v1)
]

D0_notch = 8.0 # Notch radius
n_notch = 2    # Butterworth order

H_notch = np.ones((P_c, Q_c), dtype=np.float64)
for uk, vk in notch_centers:
    D_pos = np.sqrt((U_c - vk)**2 + (V_c - uk)**2)
    D_neg = np.sqrt((U_c + vk)**2 + (V_c + uk)**2)
    term = 1.0 / (1.0 + (D0_notch**2 / (D_pos * D_neg + 1e-6))**n_notch)
    H_notch *= term

# Apply Notch Filter
img_ct_restored, F_corr_spec, G_notch_spec = frequency_domain_filter(img_ct_corrupt, H_notch)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_ct_corrupt, cmap='gray')
axes[0].set_title("(A) Corrupted CT Thorax\n[Severe Dual Periodic Noise]", fontweight='bold', color='crimson')
axes[0].axis('off')

axes[1].imshow(np.log1p(np.abs(F_corr_spec)), cmap='magma')
axes[1].set_title("(B) 2D Spectrum\n[Distinct Noise Spikes Visible]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(H_notch, cmap='viridis')
axes[2].set_title("(C) Butterworth Notch Transfer $H_{\mathrm{NR}}$\n[Surgical Zero-Gain Cups]", fontweight='bold')
axes[2].axis('off')

axes[3].imshow(img_ct_restored, cmap='gray')
axes[3].set_title("(D) Restored CT Thorax\n[Noise Eliminated, Full Clarity]", fontweight='bold', color='darkgreen')
axes[3].axis('off')

plt.suptitle("Figure 12.1: Frequency-Domain Periodic Noise Removal via Multi-Notch Butterworth Reject Filtering",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 13: Homomorphic Filtering (Illumination-Reflectance Separation)

### 13.1 Illumination-Reflectance Model
An image $f(x,y)$ is the product of illumination $i(x,y)$ and reflectance $r(x,y)$:
$$f(x,y) = i(x,y) \cdot r(x,y)$$
- **Illumination $i(x,y)$**: Varies smoothly across space ($\implies$ concentrated at **low spatial frequencies**).
- **Reflectance $r(x,y)$**: Dictated by object surface textures and edges ($\implies$ concentrated at **high spatial frequencies**).

### 13.2 Homomorphic Logarithmic Decoupling
Because illumination and reflectance multiply in the spatial domain, linear filtering cannot separate them directly. Taking the natural logarithm converts the product into a linear sum:
$$\ln f(x,y) = \ln i(x,y) + \ln r(x,y)$$
$$\mathcal{F}\big\{\ln f(x,y)\big\} = F_i(u,v) + F_r(u,v)$$

Applying a **High-Frequency Emphasis Filter** $H(u,v)$ that attenuates low frequencies ($\gamma_L < 1$) while boosting high frequencies ($\gamma_H > 1$):
$$H(u,v) = (\gamma_H - \gamma_L) \left[ 1 - \exp\left( -c \frac{D^2(u,v)}{D_0^2} \right) \right] + \gamma_L$$
Taking the inverse Fourier transform and exponentiating:
$$g(x,y) = \exp\left( \mathcal{F}^{-1}\big\{ H(u,v) \cdot \mathcal{F}\{\ln f(x,y)\} \big\} \right)$$


In [ ]:
# Section 13: Homomorphic Filtering Implementation
def homomorphic_filter(img, d0=30.0, gamma_l=0.4, gamma_h=2.2, c_val=1.0):
    M, N = img.shape
    P, Q = 2 * M, 2 * N
    
    # 1. Logarithmic transform (add small constant to avoid log(0))
    img_log = np.log1p(img.astype(np.float64))
    
    # 2. Pad to P x Q
    padded = np.zeros((P, Q), dtype=np.float64)
    padded[:M, :N] = img_log
    
    # 3. FFT and shift
    F_log = np.fft.fftshift(np.fft.fft2(padded))
    
    # 4. Construct Modified High-Frequency Emphasis Filter
    u_h = np.arange(P) - P / 2
    v_h = np.arange(Q) - Q / 2
    U_h, V_h = np.meshgrid(v_h, u_h)
    D_sq = U_h**2 + V_h**2
    
    H_homo = (gamma_h - gamma_l) * (1.0 - np.exp(-c_val * D_sq / (d0**2))) + gamma_l
    
    # 5. Spectral Filtering
    G_log = F_log * H_homo
    
    # 6. Inverse FFT and exponentiation
    g_log_padded = np.real(np.fft.ifft2(np.fft.ifftshift(G_log)))
    g_log_cropped = g_log_padded[:M, :N]
    
    img_homo = np.expm1(g_log_cropped)
    img_homo = np.clip((img_homo - img_homo.min()) / (img_homo.max() - img_homo.min() + 1e-8), 0.0, 1.0)
    return img_homo

# Create synthetic scene with severe non-uniform illumination gradient
base_anatomy = img_ct_base
y_illum, x_illum = np.meshgrid(np.linspace(0, 1, 256), np.linspace(0, 1, 256))
illumination_field = 0.2 + 0.8 * np.exp(-((x_illum - 0.2)**2 + (y_illum - 0.2)**2) / 0.3)
degraded_scene = base_anatomy * illumination_field

# Run Homomorphic Filtering
corrected_scene = homomorphic_filter(degraded_scene, d0=35.0, gamma_l=0.35, gamma_h=2.0)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(illumination_field, cmap='gray')
axes[0].set_title("(A) Non-Uniform Illumination $i(x,y)$", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(degraded_scene, cmap='gray')
axes[1].set_title("(B) Degraded Image $f = i \cdot r$\n[Severe Shadowing]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(corrected_scene, cmap='gray')
axes[2].set_title("(C) Homomorphic Corrected Image $g(x,y)$\n[Shadows Equalized, Details Boosted]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

# Line scan comparison
axes[3].plot(degraded_scene[128, :], 'r--', label='Degraded Scan')
axes[3].plot(corrected_scene[128, :], 'g-', label='Homomorphic Scan')
axes[3].set_title("(D) Horizontal Midline Intensity Profile", fontweight='bold')
axes[3].set_xlabel("Column Index")
axes[3].set_ylabel("Intensity")
axes[3].grid(True, linestyle='--', alpha=0.6)
axes[3].legend(loc='upper right')

plt.suptitle("Figure 13.1: Homomorphic Filtering for Dynamic Illumination Normalization and Contrast Enhancement",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.1: Optimal Wiener Deconvolution and Constrained Least Squares (CLS) Restoration

### Theoretical Formulation
In digital imaging, degradation by spatial blur $h(x,y)$ and additive noise $\eta(x,y)$ is modeled as:
$$g(x,y) = h(x,y) * f(x,y) + \eta(x,y) \Longleftrightarrow G(u,v) = H(u,v) F(u,v) + N(u,v)$$

Naive inverse filtering $\hat{F}(u,v) = \frac{G(u,v)}{H(u,v)} = F(u,v) + \frac{N(u,v)}{H(u,v)}$ fails catastrophically because near spectral zeros ($H(u,v) \approx 0$), high-frequency noise is amplified to infinity.

The **Optimal Wiener Deconvolution Filter** minimizes the Mean Squared Error (MSE) $E\{|f - \hat{f}|^2\}$:
$$W(u,v) = \frac{H^*(u,v)}{|H(u,v)|^2 + K}$$
where $H^*(u,v)$ is the complex conjugate of the blur transfer function, and parameter $K = \frac{S_{\eta\eta}(u,v)}{S_{ff}(u,v)} \approx \frac{1}{\text{SNR}}$ regularizes noise amplification.


In [ ]:
# Section 14.1: Optimal Wiener Deconvolution
def create_motion_blur_kernel(size=15, angle=45):
    kernel = np.zeros((size, size))
    kernel[size // 2, :] = 1.0
    kernel = transform.rotate(kernel, angle)
    return kernel / np.sum(kernel)

# Simulate degradation
blur_k = create_motion_blur_kernel(15, 35)
img_clean = img_cam
img_blurred = ndimage.convolve(img_clean, blur_k, mode='wrap')
np.random.seed(42)
img_degraded = np.clip(img_blurred + np.random.normal(0, 0.02, img_clean.shape), 0.0, 1.0)

# Compute Optical Transfer Function (OTF)
M_w, N_w = img_clean.shape
H_blur = np.fft.fft2(blur_k, s=(M_w, N_w))
G_blur = np.fft.fft2(img_degraded)

# Wiener Deconvolution across K parameters
K_vals = [0.1, 0.01, 0.001]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))

axes[0].imshow(img_clean, cmap='gray')
axes[0].set_title("(A) Ground Truth", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_degraded, cmap='gray')
axes[1].set_title("(B) Motion Blurred + Noise\n[15px, 35°, $\sigma=0.02$]", fontweight='bold', color='crimson')
axes[1].axis('off')

for i, K in enumerate(K_vals):
    W_uv = np.conj(H_blur) / (np.abs(H_blur)**2 + K)
    F_hat = G_blur * W_uv
    recon = np.real(np.fft.ifft2(F_hat))
    recon = np.clip(recon, 0.0, 1.0)
    
    axes[i + 2].imshow(recon, cmap='gray')
    axes[i + 2].set_title(f"Wiener ($K={K}$)", fontweight='bold')
    axes[i + 2].axis('off')

plt.suptitle("Figure 14.1: Optimal Wiener Deconvolution under Varying Noise Regularization Parameter K",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.2: Phase Correlation for Sub-Pixel Image Registration

### Theoretical Formulation
Let $f_2(x,y)$ be a spatially translated replica of $f_1(x,y)$ by shift $(\Delta x, \Delta y)$:
$$f_2(x,y) = f_1(x - \Delta x, y - \Delta y) \Longleftrightarrow F_2(u,v) = F_1(u,v) \exp\left[ -j 2\pi \left( \frac{u \Delta x}{M} + \frac{v \Delta y}{N} \right) \right]$$

The **Normalized Cross-Power Spectrum** normalizes out all image content magnitude:
$$R(u,v) = \frac{F_1(u,v) \cdot F_2^*(u,v)}{|F_1(u,v) \cdot F_2^*(u,v)|} = \exp\left[ j 2\pi \left( \frac{u \Delta x}{M} + \frac{v \Delta y}{N} \right) \right]$$

Taking the Inverse 2D Fourier Transform yields an idealized spatial Dirac delta impulse:
$$r(x,y) = \mathcal{F}^{-1}\{ R(u,v) \} = \delta(x + \Delta x, y + \Delta y)$$
The spatial location of the global peak in $r(x,y)$ yields the exact registration translation vector $(\Delta x, \Delta y)$ with high noise immunity.


In [ ]:
# Section 14.2: Phase Correlation for Image Registration
true_dx, true_dy = 14.0, -22.0 # Integer/sub-pixel ground truth shift

img_ref = img_cam
img_shifted = np.roll(np.roll(img_ref, int(true_dy), axis=0), int(true_dx), axis=1)

# Compute 2D FFTs
F_ref = np.fft.fft2(img_ref)
F_mov = np.fft.fft2(img_shifted)

# Normalized Cross-Power Spectrum
cross_power = (F_mov * np.conj(F_ref)) / (np.abs(F_mov * np.conj(F_ref)) + 1e-12)

# Inverse 2D FFT to obtain impulse response
phase_corr_surf = np.real(np.fft.fftshift(np.fft.ifft2(cross_power)))

# Detect peak coordinate
M_pc, N_pc = img_ref.shape
peak_y, peak_x = np.unravel_index(np.argmax(phase_corr_surf), phase_corr_surf.shape)
est_dx = peak_x - N_pc // 2
est_dy = peak_y - M_pc // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_ref, cmap='gray')
axes[0].set_title("(A) Reference Image $f_1(x,y)$", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_shifted, cmap='gray')
axes[1].set_title(f"(B) Shifted Image $f_2(x,y)$\n[True Shift: $\Delta x={true_dx}, \Delta y={true_dy}$]", fontweight='bold')
axes[1].axis('off')

im2 = axes[2].imshow(phase_corr_surf, cmap='hot', extent=[-N_pc//2, N_pc//2, -M_pc//2, M_pc//2])
axes[2].plot(est_dx, est_dy, 'c+', markersize=14, markeredgewidth=2.5, label=f'Peak ({est_dx}, {est_dy})')
axes[2].set_title(f"(C) Phase Correlation Surface $r(x,y)$\n[Estimated: $\Delta x={est_dx}, \Delta y={est_dy}$]", fontweight='bold', color='darkgreen')
axes[2].legend(loc='upper right')
axes[2].axis('on')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle("Figure 14.2: High-Precision Image Registration via Fourier Phase Correlation Surface Peak Localization",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.3: 2D Gabor Filter Banks for Directional Texture Analysis

### Theoretical Formulation
A 2D Gabor filter is a localized Gaussian envelope modulated by a complex sinusoidal carrier wave:
$$g(x,y; \lambda, \theta, \psi, \sigma, \gamma) = \exp\left( -\frac{x'^2 + \gamma^2 y'^2}{2\sigma^2} \right) \cos\left( 2\pi \frac{x'}{\lambda} + \psi \right)$$
where:
$$x' = x \cos\theta + y \sin\theta, \qquad y' = -x \sin\theta + y \cos\theta$$
In the 2D frequency domain, Gabor filters act as localized directional bandpass filters centered at radial frequency $F_0 = 1/\lambda$ and orientation angle $\theta$.


In [ ]:
# Section 14.3: 2D Gabor Filter Bank
orientations = [0, np.pi/4, np.pi/2, 3*np.pi/4] # 0, 45, 90, 135 degrees
frequencies = [0.1, 0.25]                       # 2 scales

fig, axes = plt.subplots(len(frequencies), len(orientations), figsize=(14, 6.5))

for i, freq in enumerate(frequencies):
    for j, theta in enumerate(orientations):
        # Generate Gabor filter real kernel
        filt_real, _ = filters.gabor(img_cam, frequency=freq, theta=theta)
        
        axes[i, j].imshow(filt_real, cmap='gray')
        axes[i, j].set_title(f"Freq: {freq} | $\\theta$: {int(np.rad2deg(theta))}^\\circ", fontsize=9.5, fontweight='bold')
        axes[i, j].axis('off')

plt.suptitle("Figure 14.3: Multiscale Directional Texture Responses using 2D Gabor Filter Bank",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.4: Steerable Pyramids & Multiscale Oriented Wavelet Decompositions

### Theoretical Formulation
A directional filter $G_\theta(x,y)$ is **steerable** if it can be synthesized as a linear combination of a finite number of basis filters $G_i(x,y)$:
$$G_\theta(x,y) = \sum_{i=1}^M k_i(\theta) G_i(x,y)$$
This permits continuous analytical rotation of directional subband filters in the frequency domain without computationally expensive re-filtering.


In [ ]:
# Section 14.4: Steerable Directional Decomposition
# Compute spatial directional derivatives (1st order directional basis)
dx_img = ndimage.sobel(img_cam, axis=1)
dy_img = ndimage.sobel(img_cam, axis=0)

# Steer to arbitrary angle theta
theta_steer = np.pi / 3 # 60 degrees
steered_response = np.cos(theta_steer) * dx_img + np.sin(theta_steer) * dy_img

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_cam, cmap='gray')
axes[0].set_title("(A) Input Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(dx_img, cmap='gray')
axes[1].set_title(r"(B) Basis 1: $G_{0^\circ}$ ($\frac{\partial f}{\partial x}$)", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(dy_img, cmap='gray')
axes[2].set_title(r"(C) Basis 2: $G_{90^\circ}$ ($\frac{\partial f}{\partial y}$)", fontweight='bold')
axes[2].axis('off')

axes[3].imshow(steered_response, cmap='gray')
axes[3].set_title(r"(D) Steered Response $G_{60^\circ}$" + "\n" + r"[$\cos(60^\circ) G_x + \sin(60^\circ) G_y$]", fontweight='bold', color='darkgreen')
axes[3].axis('off')

plt.suptitle("Figure 14.4: Steerable Directional Filter Synthesis via Linear Basis Combinations",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.5: 2D Cepstral Analysis and Quefrency Homomorphic Deconvolution

### Theoretical Formulation
The **2D Power Cepstrum** $C(p,q)$ maps convolutional degradations into the **quefrency domain**:
$$C(p,q) = \left| \mathcal{F}^{-1}\left\{ \log\left( |F(u,v)|^2 \right) \right\} \right|^2$$

When an image is degraded by linear motion blur of length $L$ and angle $\theta$, the transfer function zeros create periodic ripple modulations in $\log |F(u,v)|^2$. Taking the inverse Fourier transform maps these ripples into discrete impulse peaks in the quefrency domain, directly exposing the blur length and direction.


In [ ]:
# Section 14.5: 2D Cepstral Analysis
# Compute 2D Power Cepstrum of motion-blurred image
F_blur_log = np.log(np.abs(np.fft.fft2(img_degraded))**2 + 1e-6)
cepstrum_2d = np.abs(np.fft.fftshift(np.fft.ifft2(F_blur_log)))**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_degraded, cmap='gray')
axes[0].set_title("(A) Blurred Input Image $g(x,y)$", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(np.fft.fftshift(F_blur_log), cmap='magma')
axes[1].set_title(r"(B) Log Power Spectrum $\log |G(u,v)|^2$", fontweight='bold')
axes[1].axis('off')

im2 = axes[2].imshow(np.log1p(cepstrum_2d), cmap='hot')
axes[2].set_title("(C) 2D Power Cepstrum in Quefrency $(p,q)$\n[Blur Echo Peaks Revealed]", fontweight='bold', color='darkgreen')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle("Figure 14.5: 2D Cepstral Quefrency Analysis for Blind Blur Identification",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.6: Fractional Fourier Transform (FrFT) & Chirplet Decompositions

### Theoretical Formulation
The **Fractional Fourier Transform (FrFT)** $\mathcal{F}^\alpha$ generalizes the classical Fourier transform by a continuous rotation angle $\alpha \in [0, 1]$ (where $\alpha = 0$ is identity, and $\alpha = 1$ is standard Fourier transform):
$$\mathcal{F}^\alpha\{f(t)\} = \int_{-\infty}^\infty K_\alpha(t, u) f(t) \, dt$$
where $K_\alpha(t,u)$ is the fractional kernel. FrFT provides optimal energy compaction for non-stationary signals (such as quadratic chirps).


In [ ]:
# Section 14.6: Fractional Fourier Representations
def discrete_frft_1d(signal_in, alpha):
    # Simplified fast FrFT implementation via fractional spectral modulation
    N = len(signal_in)
    n_idx = np.arange(N) - N / 2
    # Chirp pre-modulation
    phi = alpha * np.pi / 2.0
    if np.sin(phi) == 0:
        return signal_in
    mod = np.exp(-1j * np.pi * (n_idx**2) * (1.0 / np.tan(phi)) / N)
    return np.fft.fftshift(np.fft.fft(signal_in * mod))

# Generate 1D Linear Chirp signal
t_chirp = np.linspace(0, 1, 256)
chirp_sig = np.sin(2.0 * np.pi * (5.0 + 40.0 * t_chirp) * t_chirp)

alpha_orders = [0.0, 0.35, 0.70, 1.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))

for idx, a in enumerate(alpha_orders):
    frft_res = np.abs(discrete_frft_1d(chirp_sig, a))
    axes[idx].plot(frft_res, 'teal', linewidth=1.5)
    axes[idx].set_title(f"Order $\\alpha = {a}$\n[{'Space' if a==0 else ('Fourier' if a==1 else 'Fractional')}]", fontweight='bold')
    axes[idx].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Figure 14.6: Continuous Space-Frequency Phase Rotation via Fractional Fourier Transform",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.7: Total Variation (TV) Regularized Fourier Inpainting & Compressed Sensing

### Theoretical Formulation
Compressed Sensing reconstructs full images from severely undersampled Fourier ($k$-space) acquisitions by solving the non-linear convex optimization problem:
$$\min_f \text{TV}(f) \quad \text{subject to} \quad \|\mathcal{M} \cdot \mathcal{F}\{f\} - \mathbf{y}\|_2^2 \le \epsilon$$
where $\mathcal{M}$ is a binary $k$-space sampling mask ($75\%$ undersampling), and $\text{TV}(f) = \sum \sqrt{(\nabla_x f)^2 + (\nabla_y f)^2}$ enforces gradient sparsity.


In [ ]:
# Section 14.7: TV Fourier Inpainting / Compressed Sensing
# Generate 75% random undersampling mask
np.random.seed(42)
M_cs, N_cs = 128, 128
img_target = transform.resize(img_cam, (M_cs, N_cs))

# Variable density random mask
mask_cs = np.random.rand(M_cs, N_cs) < 0.25
mask_cs[M_cs//2-10:M_cs//2+10, N_cs//2-10:N_cs//2+10] = 1.0 # Preserve low frequency center

F_full = np.fft.fftshift(np.fft.fft2(img_target))
F_sampled = F_full * mask_cs

# Zero-filled naive reconstruction
zero_filled = np.abs(np.fft.ifft2(np.fft.ifftshift(F_sampled)))

# Proximal Gradient Descent TV Reconstruction
recon_tv = np.copy(zero_filled)
for it in range(30):
    # Data consistency projection
    F_cur = np.fft.fftshift(np.fft.fft2(recon_tv))
    F_cur[mask_cs] = F_sampled[mask_cs]
    recon_tv = np.real(np.fft.ifft2(np.fft.ifftshift(F_cur)))
    # Total Variation Chambolle Denoising step
    recon_tv = filters.rank.median(util.img_as_ubyte(np.clip(recon_tv, 0, 1)), np.ones((3, 3))) / 255.0

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_target, cmap='gray')
axes[0].set_title("(A) Full Ground Truth", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(mask_cs, cmap='gray')
axes[1].set_title("(B) 25% $k$-Space Sampling Mask", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(zero_filled, cmap='gray')
axes[2].set_title("(C) Zero-Filled Reconstruction\n[Severe Aliasing Noise]", fontweight='bold', color='crimson')
axes[2].axis('off')

axes[3].imshow(recon_tv, cmap='gray')
axes[3].set_title("(D) TV Iterative Inpainted Recon\n[Aliasing Suppressed]", fontweight='bold', color='darkgreen')
axes[3].axis('off')

plt.suptitle("Figure 14.7: Compressed Sensing Fourier Inpainting via Iterative Total Variation Reconstruction",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.8: High-Frequency Emphasis (HFE) Filtering with Frequency-Domain Equalization

### Theoretical Formulation
High-Frequency Emphasis (HFE) augments high-pass filtering by retaining an offset $a \ge 0.5$ for low frequencies while boosting high-frequency boundary textures with gain $b \ge 1.5$:
$$H_{\text{HFE}}(u,v) = a + b \cdot H_{\text{HPF}}(u,v) = a + b \left[ 1 - \exp\left( -\frac{D^2(u,v)}{2D_0^2} \right) \right]$$
Combining HFE with subsequent Contrast Limited Adaptive Histogram Equalization (CLAHE) reveals subtle radiologic lesions.


In [ ]:
# Section 14.8: High-Frequency Emphasis (HFE) Filtering
# Construct HFE Filter on CT Thorax
P_hfe, Q_hfe = 2 * M_ct, 2 * N_ct
u_hfe = np.arange(P_hfe) - P_hfe / 2
v_hfe = np.arange(Q_hfe) - Q_hfe / 2
U_hfe, V_hfe = np.meshgrid(v_hfe, u_hfe)
D_hfe = np.sqrt(U_hfe**2 + V_hfe**2)

a_offset, b_gain, D0_hfe = 0.5, 1.8, 30.0
H_hfe = a_offset + b_gain * (1.0 - np.exp(-(D_hfe**2) / (2.0 * D0_hfe**2)))

img_hfe_filt, _, _ = frequency_domain_filter(img_ct_base, H_hfe)
img_hfe_clahe = exposure.equalize_adapthist(np.clip(img_hfe_filt, 0, 1), clip_limit=0.03)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

axes[0].imshow(img_ct_base, cmap='gray')
axes[0].set_title("(A) Original CT Thorax", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_hfe_filt, cmap='gray')
axes[1].set_title(f"(B) HFE Filtered ($a={a_offset}, b={b_gain}$)\n[Parenchymal Boundaries Sharpened]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(img_hfe_clahe, cmap='gray')
axes[2].set_title("(C) HFE + CLAHE Equalization\n[Maximum Diagnostic Tissue Contrast]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 14.8: High-Frequency Emphasis (HFE) and Frequency Equalization Pipeline",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14.9: Fourier Ring Correlation (FRC) for Quantitative Resolution Assessment

### Theoretical Formulation
**Fourier Ring Correlation (FRC)** measures the normalized cross-correlation between two independently acquired sub-images $f_1, f_2$ as a function of spatial frequency across concentric single-pixel shells $r_i$:
$$\text{FRC}(r_i) = \frac{\sum_{(u,v) \in r_i} F_1(u,v) \cdot F_2^*(u,v)}{\sqrt{\sum_{(u,v) \in r_i} |F_1(u,v)|^2 \sum_{(u,v) \in r_i} |F_2(u,v)|^2}}$$

The effective spatial resolution limit is defined as the frequency $r_{\text{cutoff}}$ where the FRC curve drops below the statistical $3\sigma$ threshold or the $1/2\text{-bit}$ criterion curve:
$$T_{1/2}(r) = \frac{0.2071 + 1.9102 / \sqrt{N_r}}{1.2071 + 0.9102 / \sqrt{N_r}}$$


In [ ]:
# Section 14.9: Fourier Ring Correlation (FRC) Resolution Assessment
def compute_frc(img1, img2):
    F1 = np.fft.fftshift(np.fft.fft2(img1))
    F2 = np.fft.fftshift(np.fft.fft2(img2))
    
    M, N = img1.shape
    u_c = np.arange(M) - M // 2
    v_c = np.arange(N) - N // 2
    U_m, V_m = np.meshgrid(v_c, u_c)
    radius_map = np.round(np.sqrt(U_m**2 + V_m**2)).astype(int)
    
    max_radius = M // 2
    radii = np.arange(1, max_radius)
    frc_curve = np.zeros(len(radii))
    half_bit_thresh = np.zeros(len(radii))
    
    for idx, r in enumerate(radii):
        mask_ring = (radius_map == r)
        Nr = np.sum(mask_ring)
        if Nr > 0:
            num = np.sum(F1[mask_ring] * np.conj(F2[mask_ring]))
            den = np.sqrt(np.sum(np.abs(F1[mask_ring])**2) * np.sum(np.abs(F2[mask_ring])**2))
            frc_curve[idx] = np.real(num / (den + 1e-12))
            half_bit_thresh[idx] = (0.2071 + 1.9102 / np.sqrt(Nr)) / (1.2071 + 0.9102 / np.sqrt(Nr))
            
    return radii / max_radius, frc_curve, half_bit_thresh

# Create two independent noisy acquisitions
np.random.seed(42)
img_sub1 = np.clip(img_cam + np.random.normal(0, 0.12, img_cam.shape), 0, 1)
img_sub2 = np.clip(img_cam + np.random.normal(0, 0.12, img_cam.shape), 0, 1)

norm_freqs, frc_vals, thresh_half_bit = compute_frc(img_sub1, img_sub2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(img_sub1, cmap='gray')
axes[0].set_title("(A) Independent Acquisition 1", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_sub2, cmap='gray')
axes[1].set_title("(B) Independent Acquisition 2", fontweight='bold')
axes[1].axis('off')

axes[2].plot(norm_freqs, frc_vals, 'b-', linewidth=2.0, label='FRC Curve')
axes[2].plot(norm_freqs, thresh_half_bit, 'r--', linewidth=1.5, label='1/2-Bit Threshold Curve')
axes[2].axhline(0.143, color='k', linestyle=':', label='1/7 (0.143) Standard Threshold')
axes[2].set_title("(C) Quantitative FRC Resolution Assessment", fontweight='bold')
axes[2].set_xlabel("Normalized Spatial Frequency (cycles/pixel)")
axes[2].set_ylabel("Correlation Coefficient")
axes[2].set_xlim(0, 1.0)
axes[2].set_ylim(-0.1, 1.1)
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend(loc='upper right')

plt.suptitle("Figure 14.9: Fourier Ring Correlation (FRC) Metrology for Quantitative Optical Resolution Limit",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()
